## The Conversation Around Tabular Data Is Back. The Tables Are Still Messy. 
### A hands-on walkthrough of using skrub to turn a messy sales leads dataset into model-ready data.

> This notebook is a companion to the article of the same name. You can read the article straight through, or follow along here with the code, whichever works best for you. Companion notebooks are also available in marimo and Kaggle formats.

---

The rise of tabular foundation models has brought the spotlight back to tabular data. And yet, regardless of whether you're training gradient boosting models or experimenting with TFMs, one thing has remained unchanged and that is the messy data.

In this notebook, we'll walk through a complete tabular workflow using **skrub**, from inspecting a messy sales leads dataset to building a model-ready scikit-learn pipeline. The task is binary classification: predict whether a sales lead converted into a customer.

**What we'll cover:**
1. Loading and inspecting the data
2. Exploring with `TableReport`
3. Cleaning with `Cleaner`
4. Joining external data with `fuzzy_join()`
5. Consolidating messy categories with `deduplicate()`
6. Feature engineering with `TableVectorizer`
7. Building a predictive pipeline with `tabular_pipeline`

## 0. Installation & Setup

First things first, let's start by installing skrub:

In [ ]:
# Uncomment to install
# !pip install skrub -U

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

from skrub import (
    Cleaner,
    TableReport,
    TableVectorizer,
    deduplicate,
    fuzzy_join,
    tabular_pipeline,
)

## 1. Loading the data

We'll work with a synthetic sales leads dataset. The objective is to predict whether a lead eventually converted into a customer. The target column, `converted`, records this observed outcome — `1` for converted, `0` otherwise.

Let's load the dataset and take a quick look at it. We'll separate the target from the input features and also drop `lead_id`, since it simply acts as an identifier.

In [ ]:
leads = pd.read_csv("data/messy_sales_leads.csv")

conversion_target = leads["converted"]
lead_features = leads.drop(columns=["converted", "lead_id"])

lead_features.head().T

,0,1,2,3,4
company_name,Acme Group,Nova Systems,Nova Analytics,Nimbus Labs,Nova Analytics
industry_code,TRVL,MEDIA,TRVL,TECH,TECH
industry_name,Trave,Media,Travel,Technology,Technology
lead_source,Webinar,NaN,Cold email,Website,Referral
product_interest,Security audit,CRM integration,Security audit,Data warehouse,Analytics platform
inquiry_text,Looking for security audit for a operations te...,Needs help replacing legacy BI tool with crm i...,Looking for security audit for a operations te...,Comparing vendors for data warehouse after a f...,Comparing vendors for analytics platform after...
office_country,France,Brasil,Canada,Canada,Espana
created_at,NaN,2022-03-25,2023-10-15,2021-07-04,2021-05-04
last_contacted_at,2023-07-07,2022-10-07,2023-02-06,2021-01-05,2021-12-19
company_size,79,257,27,69,114


Even from the first few rows, we can already spot several potential issues:
- Missing values in multiple columns
- Date columns stored as strings (`created_at`, `last_contacted_at`)
- Misspelled industry names (`Trave`, `Retial`)
- Country name variants (`Brasil`, `Espana`, `Great Britain`)
- A column containing almost the same value throughout (`constant_source`)

## 2. Exploring the data

The above static preview can only tell us so much. Skrub's `TableReport` combines several exploratory views into a single interactive report. Alongside summary statistics and column distributions, it also shows relationships between variables using measures such as Pearson correlation and Cramér's V.

In [92]:
TableReport(lead_features)

Processing column  16 / 16


,,,,,,,,,,,,,,,,


You can click on any cell in the generated table and get detailed information about the corresponding column, including missing values, the number of unique values, value distributions, and the most common categories.

If you're working outside a notebook environment, you can also open the report directly in your browser, or export it as an HTML file to share with colleagues:

In [114]:
# TableReport(lead_features).open()
# TableReport(lead_features).write_html("lead_features_report.html")

## 3. Pre-processing the data

Now that we have a better understanding of the dataset, it's time to clean things up. Some issues can be fixed automatically, while others require a bit more context and standardization.

### 3.1 Cleaning common data issues with Cleaner

`Cleaner` is a scikit-learn-compatible transformer that automates several routine preprocessing tasks: parsing date columns correctly, cleaning null strings, dropping uninformative columns, and casting near-constant columns to strings.

It is important to mention that while `Cleaner` standardizes null-like values and ensures they are treated consistently across the dataframe, it does **not** perform imputation. Missing-value handling strategies can be applied later depending on the modeling workflow.

In [94]:
cleaned_features = Cleaner(drop_if_constant=True).fit_transform(lead_features)
cleaned_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   company_name       2500 non-null   object        
 1   industry_code      2500 non-null   object        
 2   industry_name      2445 non-null   object        
 3   lead_source        2312 non-null   object        
 4   product_interest   2500 non-null   object        
 5   inquiry_text       2500 non-null   object        
 6   office_country     2500 non-null   object        
 7   created_at         2420 non-null   datetime64[ns]
 8   last_contacted_at  2353 non-null   datetime64[ns]
 9   company_size       2500 non-null   int64         
 10  annual_revenue     2415 non-null   float64       
 11  lead_score         2430 non-null   float64       
 12  pages_viewed       2500 non-null   int64         
 13  requested_demo     2500 non-null   int64         
 14  sales_no

As we can see, the datetime columns have now been correctly parsed, and the `constant_source` column has been removed automatically.

### 3.2 Joining external data with fuzzy_join

More often than not, useful data is spread across multiple tables. In our case, each lead has an `office_country` value but that alone doesn't tell us much about the broader market. We'll bring in a small country reference dataset containing fields such as `market_size_score`, `digital_adoption_score`, `average_deal_value_index`, and `sales_cycle_index`.

The challenge is that country names are not written consistently — the same country may appear as `USA`, `U.S.`, or `United States` while the reference table uses `United States of America`. Similarly, `Brasil` should match `Brazil`, and `Espana` should match `Spain`.

A regular `pandas.merge()` only works when the join keys match exactly. This is where `fuzzy_join()` comes in.

In [115]:
countries = pd.read_csv("data/country_reference.csv")
countries.head()

,country_name,region,market_size_score,digital_adoption_score,average_deal_value_index,sales_cycle_index
0,United States of America,North America,97.0,94.7,117.1,74.4
1,United Kingdom,Europe,74.0,91.9,105.1,70.8
2,France,Europe,72.9,86.9,100.7,66.5
3,Germany,Europe,81.8,83.6,106.0,72.1
4,India,Asia-Pacific,85.5,76.3,76.2,60.6


In [116]:
leads_enriched = fuzzy_join(
    cleaned_features,
    countries,
    left_on="office_country",
    right_on="country_name",
)

# Check the matches
leads_enriched[["office_country", "country_name"]].drop_duplicates().sort_values(
    "country_name"
).head(20)

,office_country,country_name
36,USA,Australia
21,Australia,Australia
1,Brasil,Brazil
98,Brazil,Brazil
2,Canada,Canada
0,France,France
12,Germany,Germany
40,Deutschland,India
6,India,India
11,Italia,Italy


By default, `fuzzy_join()` matches each value to its closest counterpart in the reference table. The matching threshold can be adjusted using the `max_dist` parameter.

### 3.3 Consolidating similar categories with deduplicate

The country names problem was relatively easy to solve because we had a separate reference table. But what do you do when no such table exists? In our dataset, the `industry_name` column contains several entries that appear to describe the same category but are written slightly differently.



In [118]:
sorted(cleaned_features["industry_name"].dropna().unique())

['Education',
 'Educaton',
 'Finanace',
 'Finance',
 'Healtcare',
 'Healthcare',
 'Manufactring',
 'Manufacturing',
 'Meda',
 'Media',
 'Retail',
 'Retial',
 'Technlogy',
 'Technology',
 'Trave',
 'Travel']

Should `Education` and `Educaton` be treated as different industries? What about `Finanace` and `Finance`, or `Healtcare` and `Healthcare`? In situations like these, manually cleaning categories quickly becomes tedious.

Skrub has a built-in `deduplicate()` function for these scenarios that groups similar strings together and maps them to a cleaner category. Under the hood, it uses clustering based on string similarities.

In [119]:
cleaned_features["clean_industry_name"] = cleaned_features["industry_name"].replace(
    deduplicate(cleaned_features["industry_name"].dropna()).to_dict()
)

sorted(cleaned_features["clean_industry_name"].dropna().unique())

['Education',
 'Finance',
 'Healthcare',
 'Manufacturing',
 'Media',
 'Retail',
 'Technology',
 'Travel']

One thing to keep in mind: similar-looking categories are not always the same. This step works best when spelling variations or duplicate labels genuinely refer to the same underlying category. Otherwise, important information may be lost.

From this point onward, we'll use `model_features` as the dataframe passed to the modeling pipeline.

In [120]:
model_features = leads_enriched.copy()
model_features.head()

,company_name,industry_code,industry_name,lead_source,product_interest,inquiry_text,office_country,created_at,last_contacted_at,company_size,...,pages_viewed,requested_demo,sales_notes,clean_industry_name,country_name,region,market_size_score,digital_adoption_score,average_deal_value_index,sales_cycle_index
0,Acme Group,TRVL,Trave,Webinar,Security audit,Looking for security audit for a operations te...,France,NaT,2023-07-07,79,...,3,0,NaN,Travel,France,Europe,72.9,86.9,100.7,66.5
1,Nova Systems,MEDIA,Media,NaN,CRM integration,Needs help replacing legacy BI tool with crm i...,Brasil,2022-03-25,2022-10-07,257,...,5,1,asked about pricing,Media,Brazil,Latin America,76.1,65.1,64.2,59.5
2,Nova Analytics,TRVL,Travel,Cold email,Security audit,Looking for security audit for a operations te...,Canada,2023-10-15,2023-02-06,27,...,6,0,NaN,Travel,Canada,North America,65.7,87.1,102.9,73.3
3,Nimbus Labs,TECH,Technology,Website,Data warehouse,Comparing vendors for data warehouse after a f...,Canada,2021-07-04,2021-01-05,69,...,5,0,NaN,Technology,Canada,North America,65.7,87.1,102.9,73.3
4,Nova Analytics,TECH,Technology,Referral,Analytics platform,Comparing vendors for analytics platform after...,Espana,2021-05-04,2021-12-19,114,...,5,1,NaN,Technology,Spain,Europe,62.0,68.8,84.8,65.6


## 4. Performing feature engineering

After cleaning and enriching the dataset, the final step before modeling is to transform the mixed collection of numeric, categorical, datetime, and text columns into representations that machine learning models can work with.

Skrub's `TableVectorizer` simplifies much of this process by automatically selecting suitable transformations based on the characteristics of each column. By default:

- **Low-cardinality categorical columns** are one-hot encoded
- **High-cardinality categorical columns** are transformed using `StringEncoder`
- **Numeric columns** are passed through unchanged
- **Datetime columns** are encoded using `DatetimeEncoder`

In [122]:
vectorizer = TableVectorizer()
encoded_lead_features = vectorizer.fit_transform(model_features)

encoded_lead_features.shape

(2500, 165)

`TableVectorizer` is not a black box — the fitted object is fully inspectable. You can check how skrub classified each input column:

In [124]:
vectorizer.column_to_kind_

{'company_size': 'numeric',
 'annual_revenue': 'numeric',
 'lead_score': 'numeric',
 'pages_viewed': 'numeric',
 'requested_demo': 'numeric',
 'market_size_score': 'numeric',
 'digital_adoption_score': 'numeric',
 'average_deal_value_index': 'numeric',
 'sales_cycle_index': 'numeric',
 'created_at': 'datetime',
 'last_contacted_at': 'datetime',
 'company_name': 'low_cardinality',
 'industry_code': 'low_cardinality',
 'industry_name': 'low_cardinality',
 'lead_source': 'low_cardinality',
 'product_interest': 'low_cardinality',
 'office_country': 'low_cardinality',
 'sales_notes': 'low_cardinality',
 'clean_industry_name': 'low_cardinality',
 'country_name': 'low_cardinality',
 'region': 'low_cardinality',
 'inquiry_text': 'high_cardinality'}

In [125]:
# Inspect the full fitted vectorizer
vectorizer

,cardinality_threshold,40
,low_cardinality,OneHotEncoder..._output=False)
,high_cardinality,StringEncoder()
,numeric,PassThrough()
,datetime,DatetimeEncoder()
,specific_transformers,()
,drop_null_fraction,1.0
,drop_if_constant,False
,drop_if_unique,False
,datetime_format,None
,null_strings,None


## 5. Building a predictive pipeline

Because `TableVectorizer` follows the scikit-learn API, it can be combined with virtually any scikit-learn model using the usual pipeline utilities:

In [126]:
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier

pipeline = make_pipeline(
    TableVectorizer(),
    HistGradientBoostingClassifier(random_state=0),
)

But skrub also provides `tabular_pipeline`, a convenience function that automatically combines tabular preprocessing with a scikit-learn estimator — and adapts its defaults to the chosen model.

In [127]:
pipeline = tabular_pipeline("classification")
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tablevectorizer', ...), ('histgradientboostingclassifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,cardinality_threshold,40
,low_cardinality,ToCategorical()
,high_cardinality,StringEncoder()
,numeric,PassThrough()
,datetime,DatetimeEncoder()
,specific_transformers,()
,drop_null_fraction,1.0


In [128]:
# With a specific estimator, the pipeline adapts its preprocessing steps accordingly
logistic_pipeline = tabular_pipeline(LogisticRegression(max_iter=1000))
logistic_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tablevectorizer', ...), ('simpleimputer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,cardinality_threshold,40
,low_cardinality,OneHotEncoder..._output=False)
,high_cardinality,StringEncoder()
,numeric,PassThrough()
,datetime,DatetimeEncod...ding='spline')
,specific_transformers,()
,drop_null_fraction,1.0


In [129]:
scores = cross_validate(
    pipeline,
    model_features,
    conversion_target,
    cv=3,
    scoring="roc_auc",
)

print(f"ROC-AUC scores: {scores['test_score']}")
print(f"Mean: {scores['test_score'].mean():.3f}")

ROC-AUC scores: [0.60957138 0.64828084 0.64479058]
Mean: 0.634


In [130]:
# Fit and generate predictions
pipeline.fit(model_features, conversion_target)

predictions = pipeline.predict(model_features)
conversion_probabilities = pipeline.predict_proba(model_features)

conversion_probabilities[:5]

array([[0.24590667, 0.75409333],
       [0.07838084, 0.92161916],
       [0.73165871, 0.26834129],
       [0.95104664, 0.04895336],
       [0.08744381, 0.91255619]])

## Full workflow in one cell

Here's the complete pipeline without the explanatory cells — useful if you just want to run the whole thing end to end.

In [131]:
import pandas as pd
from sklearn.model_selection import cross_validate
from skrub import Cleaner, deduplicate, fuzzy_join, tabular_pipeline

# Load
leads = pd.read_csv("data/messy_sales_leads.csv")
conversion_target = leads.pop("converted")
lead_features = leads.drop(columns="lead_id")

# Clean
lead_features = Cleaner(drop_if_constant=True).fit_transform(lead_features)

# Deduplicate industry names
mask = lead_features["industry_name"].notna()
lead_features.loc[mask, "industry_name"] = deduplicate(
    lead_features.loc[mask, "industry_name"].reset_index(drop=True)
).array

# Fuzzy join with country reference
countries = pd.read_csv("data/country_reference.csv")
lead_features = fuzzy_join(
    lead_features,
    countries,
    left_on="office_country",
    right_on="country_name",
)

# Train and evaluate
model = tabular_pipeline("classification")
scores = cross_validate(
    model, lead_features, conversion_target, cv=3, scoring="roc_auc"
)
print(f"ROC-AUC: {scores['test_score'].mean():.3f}")

ROC-AUC: 0.632
